In [ ]:
# Notes:
# Goal A: use ephemeris information to explain the observed doppler/delay data, ensure there aren't any unmodeled sources of error.
# - TROUBLE: a key step is to calculate the expected doppler shifts of the bi-static radar (tx to moon to rx).
#   In particular, we want to know the min doppler, the max doppler, and the doppler of the sub radar point (nearest point).
#   - Prediction 1: I think min/srp/max dopplers should be symmetric!
#   - Prediction 2: the min/max radial doppler should be observed 90 degrees away from the sub radar point in the direction of motion of the sub-radar point... doesn't appear to be correct
#   - Handling light time-of-flight makes this tricky!
#   - I use three different libraries (astropy, JPL Horizons, and SPICE)... 
#     - I get three different answers
#     - NONE OF THOSE ANSWERS APPEAR TO MATCH THE DATA!
#   - Explanation 1: I have a bug.
#   - Explanation 2: The ephemeris error is too high, so the answers are dominated by noise.
#     - THIS SHOULDN'T BE THE CASE, ACCORDING TO PUBLISHED ERROR PROJECTIONS!
# Goal B: combine all the doppler/delay data into a single low-noise super-resolution map of the moon.
#   I could punt on Goal A, and go straight for Goal B by fitting a curve to the doppler/delay plot.
#   The curve empirically describes the doppler and delay needed to project points onto the surface of the moon.

# Interactive script magic

In [ ]:
_interactive = True  # Skip the next cell if you want to run in interactive (cell-by-cell) mode!

In [ ]:
_interactive = False  # Automatically override _interactive to false if this notebook is run in batch mode!

# Background

## Basic principle of Planetary Radar:
Transmit short pulses at a given frequency for distance/c time
 * pulse duration should be ... pretty short (have to sum up many pulses to deal with noise)
 * pulse interval should be longer than the delay dispersion
 * or use coded long pulse waveform (pseudo-random)
 * Receive the echos, compute the response for each cell in a doppler delay grid (doppler varies across the limbs, delay varies with distance)

## Relevant papers
 * [Radar Echoes From the Moon](https://web.archive.org/web/20081029000712/http://www.eagle.ca/~harry/ba/eme/index.htm) -- historical
 * [Planetary Radar - State-of-the-Art Review](https://www.mdpi.com/2072-4292/15/23/5605) -- 2023 overview
 * [Planetary Radar Astronomy](https://nap.nationalacademies.org/read/21729/chapter/8) -- 2015 overview
 * [Planetary Radar](https://echo.jpl.nasa.gov/asteroids/ostro_1998_encyc_ss.pdf) -- 1998 overview
 * [Planetary Delay-Doppler Radar and the Long-Code Method](https://echo.jpl.nasa.gov/asteroids/harmon.2002.long.code.pdf) -- long-code method (recommends intercode -- coherent code sequence) for underspread Moon
 * [Earth-Based Radar Observations of Venus -- PDS Archive Description](https://pds-geosciences.wustl.edu/venus/arcb_nrao-v-rtls_gbt-3-delaydoppler-v1/vrm_90xx/document/venus_radar.pdf) -- Arecibo Venus data description

# Fetch the data, install dependencies

In [ ]:
# To fetch the camras data, use something like:
#! wget -r --no-clobber --no-parent -R index.html?* https://data.camras.nl/thomas/sdr-eme/

In [ ]:
# Install dependencies
#! pip install --quiet --upgrade astropy astroquery cspyce cupy-cuda12x healpy ipympl jplephem matplotlib numpy scipy sigmf tqdm

# Imports and configuration

In [ ]:
import os
import pickle

import numpy as np
from matplotlib import pyplot as pl

from astropy import units as au
from astropy import constants as ak
from astropy import coordinates as ac
from astropy import time as at

import healpy as hp

import cupy

import scipy
import sigmf
from tqdm import tqdm

In [ ]:
# Set high quality ephemerides -- may take a while to download!
# https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440_and_de441.pdf appears to recommend de440s for our purposes:
# - de440s : 1849 to 2150 (32 MB)
# - de440 : 1550 to 2650 (114 MB)
# - de441 : -13200 to 17191(3.2 GB)
ac.solar_system_ephemeris.set("de440s")
#ac.solar_system_ephemeris.set("jpl")

# Constants

In [ ]:
DWINGELOO_LAT = 52.81214958283062 * au.degree
DWINGELOO_LON = 6.396319071523311 * au.degree
DWINGELOO_HEIGHT = 70.26 * au.m

STOCKERT_LAT = 50.56944039751571 * au.degree
STOCKERT_LON = 6.721943350231514 * au.degree
STOCKERT_HEIGHT = 434.0 * au.m

tx_location = ac.EarthLocation(lat=DWINGELOO_LAT, lon=DWINGELOO_LON, height=DWINGELOO_HEIGHT, ellipsoid="WGS84")
rx_location = ac.EarthLocation(lat=STOCKERT_LAT, lon=STOCKERT_LON, height=STOCKERT_HEIGHT, ellipsoid="WGS84")

#MOON_RADIUS = 1_738_100.0 * au.m   # Equatorial radius
MOON_RADIUS = 1_737_400.0 * au.m   # Volumetric mean radius
#MOON_RADIUS = 17_000_000.0 * au.m   # HACK HACK HACK --> range needed to explain a ~0.03 Hz doppler error.

#TX_START = (1.000011 - 11.926e-6) * au.s  # From Thomas' notebook "Stockert, correct for H-maser offset Dwingeloo (1 MHz)"
TX_START = 1.0 * au.s

DATA_ROOT = 'data/'


# File handling

In [ ]:
# Load the rx data
def loadRxTxFiles(rx_chan0_sigmf_filename):
    print(f"Loading RX file(s) {rx_chan0_sigmf_filename}")
    if 1: # Just use one polarity
        rx_chan0_sigmf_file = sigmf.sigmffile.fromfile(rx_chan0_sigmf_filename)
        rx_samples = rx_chan0_sigmf_file.read_samples().astype("complex64")
        ## Normalize by the average power.
        #rx_samples /= np.mean(np.abs(rx_samples))
    
    if 0: # Combine both polarities:
        # Chan 0 is RHCP
        rx_chan0_sigmf_file = sigmf.sigmffile.fromfile(rx_chan0_sigmf_filename)
        # Chan 0 is LHCP
        rx_chan1_sigmf_filename = rx_chan0_sigmf_filename.replace("chan0", "chan1")
        rx_chan1_sigmf_file = sigmf.sigmffile.fromfile(rx_chan1_sigmf_filename)
        rx_chan0_samples = rx_chan0_sigmf_file.read_samples().astype("complex64")
        rx_chan1_samples = rx_chan1_sigmf_file.read_samples().astype("complex64")
    
        ## Normalize by the average channel power.
        #rx_chan0_samples /= np.mean(np.abs(rx_chan0_samples))
        #rx_chan1_samples /= np.mean(np.abs(rx_chan1_samples))
        
        # From Thomas' code: Combine RHCP at 80 degrees (cross-pol), LHCP at 260 (co-pol)
        rx_samples = rx_chan0_samples + np.exp(1j * 260 * np.pi / 180) * rx_chan1_samples
    
    rx_info = rx_chan0_sigmf_file.get_global_info()
    
    if 0:  # Debug: Basic rx file info
        display(rx_chan0_sigmf_file.get_global_info())
        display(rx_chan0_sigmf_file.get_captures())
        
    sample_rate = rx_info['core:sample_rate'] / au.s
    tx_filename = rx_info['core:description']
    captures = rx_chan0_sigmf_file.get_captures()
    frequency = captures[0]['core:frequency'] * au.Hz
    #print(captures[0]['core:datetime'])
    rx_start_astrotime = at.Time(captures[0]['core:datetime'])
    #print(f"{rx_start_astrotime=}")
    rx_duration = len(rx_samples) / sample_rate
    #print(f"  {rx_duration=}")

    # Load the tx waveform data
    if "2025-06-21" in rx_chan0_sigmf_filename:
        tx_sigmf_filename = f"{DATA_ROOT}/sdr-eme/tx-2025-06-21/{tx_filename}"
    else:
        tx_sigmf_filename = f"{DATA_ROOT}/sdr-eme/tx/{tx_filename}"
    print(f"  Loading TX file {tx_sigmf_filename}")
    tx_sigmf_file = sigmf.sigmffile.fromfile(tx_sigmf_filename)
    tx_samples = tx_sigmf_file.read_samples().astype("complex64")
    tx_duration = len(tx_samples) / sample_rate
    #print(f"    {tx_duration=}")
    
    if 0:  # Debug: Basic tx file info
        tx_info = tx_sigmf_file.get_global_info()
        display(tx_info)
        display(len(tx_samples))

    return rx_samples, tx_samples, sample_rate, frequency, rx_start_astrotime

if _interactive:
    # Large data files
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_32_39_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # 30s tx, Goodish
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_33_55_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # Goodish, feathery side-lobes
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_35_05_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # Goodish, the 'overspread' images to left and right seem to be blurred in delay?
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_36_21_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # 30s tx, BEST, most manually examined image... really the best data from the 03 11 datasets
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_37_18_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # Weird stuttery echoes??
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-03-11/stockert_eme_2025_03_11_19_41_04_1297.500MHz_1.00Msps_ci16_le.chan0.sigmf-meta"  # 2s tx, very poor doppler resolution, not sure why

    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-06-21/stockert_eme_2025_06_21_08_47_23_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta" # 6s rx_duration
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-06-21/stockert_eme_2025_06_21_09_06_52_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta"  # 60s tx, 66s rx!
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-06-21/stockert_eme_2025_06_21_09_06_52_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta"  # 60s tx, 66s rx!
    rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-06-21/stockert_eme_2025_06_21_10_06_39_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta"  # 60s tx, 66s rx!
    #rx_chan0_sigmf_filename = f"{DATA_ROOT}/sdr-eme/rx-2025-06-21/stockert_eme_2025_06_21_11_18_31_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta"
    #rx_chan0_sigmf_filename = f"{DATA_PREFIX}/stockert_eme_2025_06_21_11_30_02_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta" # BPSK
    rx_samples, tx_samples, sample_rate, frequency, rx_start_astrotime = loadRxTxFiles(rx_chan0_sigmf_filename)

In [ ]:
if 0: # Debug: rx spectrogram
    %matplotlib widget
    pl.figure()
    Sxx_rx, f_rx, t_rx, image_rx = pl.specgram(rx_samples, Fs=sample_rate, NFFT=2**12, clim=[-110, -80])
    pl.title('RX Spectrogram')
    pl.gca().set_xlabel("Time (s)")
    pl.gca().set_ylabel("Frequency from center (Hz)")
    pl.gca().axvline(transmission_start, color="white", linestyle="--")
    pl.gca().axvline(transmission_start + expected_delay, color="white", linestyle="--")

In [ ]:
if 0: # Debug: tx spectrogram
    nfft = nperseg = 2**8
    Sxx_tx, f_tx, t_tx, image_tx = pl.specgram(tx_samples, Fs=sample_rate, NFFT=nfft)
    pl.title('TX Spectrogram')
    pl.gca().set_xlabel("Time (s)")
    pl.gca().set_ylabel("Frequency from center (Hz)")

In [ ]:
import math

def calculate_delta_point(lat1, lon1, lat2, lon2, angular_distance):
    """
    Calculate the point that is angular distance away from point 1 in the direction of point 2.

    Args:
        lat1, lon1: Latitude and longitude of the starting point
        lat2, lon2: Latitude and longitude of the reference point
        delta: angular distance

    Returns:
        tuple: (latitude, longitude) of the point
    """

    # Convert to radians
    lat1_rad = lat1.to(au.rad).value
    lon1_rad = lon1.to(au.rad).value
    lat2_rad = lat2.to(au.rad).value
    lon2_rad = lon2.to(au.rad).value

    # Calculate the initial bearing from point 1 to point 2
    delta_lon = lon2_rad - lon1_rad
    y = math.sin(delta_lon) * math.cos(lat2_rad)
    x = (math.cos(lat1_rad) * math.sin(lat2_rad) -
         math.sin(lat1_rad) * math.cos(lat2_rad) * math.cos(delta_lon))
    bearing = math.atan2(y, x)
    angular_distance_rad = angular_distance.to(au.rad).value
    # Calculate the destination point using the spherical law of cosines
    lat3_rad = math.asin(
        math.sin(lat1_rad) * math.cos(angular_distance_rad) +
        math.cos(lat1_rad) * math.sin(angular_distance_rad) * math.cos(bearing)
    )
    lon3_rad = lon1_rad + math.atan2(
        math.sin(bearing) * math.sin(angular_distance_rad) * math.cos(lat1_rad),
        math.cos(angular_distance_rad) - math.sin(lat1_rad) * math.sin(lat3_rad)
    )
    # Convert back to degrees
    lat3 = math.degrees(lat3_rad) * au.deg
    lon3 = math.degrees(lon3_rad) * au.deg
    return lat3, lon3

if 0:
    def haversine_distance(lat1, lon1, lat2, lon2):
        """Calculate great circle distance in degrees"""
        lat1_rad, lon1_rad = lat1.to(au.rad).value, lon1.to(au.rad).value
        lat2_rad, lon2_rad = lat2.to(au.rad).value, lon2.to(au.rad).value
        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad
        a = (math.sin(dlat/2)**2 +
             math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2)
        c = 2 * math.asin(math.sqrt(a))
        return math.degrees(c)
        
    # Test case: Start at (0°, 0°) and go towards (0°, 1°) - eastward direction
    lat1, lon1 = 0.0 * au.deg, 0.0 * au.deg  # Equator, Prime Meridian
    lat2, lon2 = 0.0 * au.deg, 1.0 * au.deg  # Slightly east

    delta_deg = 90 * au.deg
    result_lat, result_lon = calculate_delta_point(lat1, lon1, lat2, lon2, delta_deg)
    print(f"Starting point: ({lat1}, {lon1})")
    print(f"Reference point: ({lat2}, {lon2})")
    print(f"Point {delta_deg} away: ({result_lat}, {result_lon})")
    distance = haversine_distance(lat1, lon1, result_lat, result_lon)
    print(f"Verification: Distance from start to result = {distance:.6f}°")

    # Test case: Start at New York and go towards London direction 90 degress
    ny_lat, ny_lon = 40.7128 * au.deg, -74.0060 * au.deg  # New York City
    london_lat, london_lon = 51.5074 * au.deg, -0.1278 * au.deg  # London

    result_lat, result_lon = calculate_delta_point(ny_lat, ny_lon, london_lat, london_lon, delta_deg)
    print(f"Starting point: New York ({ny_lat}, {ny_lon})")
    print(f"Reference point: London ({london_lat}, {london_lon})")
    print(f"Point {delta_deg} away: ({result_lat}, {result_lon})")

    distance = haversine_distance(ny_lat, ny_lon, result_lat, result_lon)
    print(f"Verification: Distance from start to result = {distance:.6f}°")

In [ ]:
# More complex doppler calculation, try to consider all the light travel times and the sub radar position on the surface of the moon.
def moonDopplerAndDelay_astropy(rx_time, tx_location, rx_location):
    # We'll use the doppler to correct the rx_samples, so all calculations are relative to rx_time.
    # We need three ICRS positions and velocities:
    #  - rx_location at rx_time
    #  - moon sub radar point at the time of a bounce that is received at rx_time
    #  - tx_location at the time of a transmission that bounces and then is received at rx_time
    rx_gcrs = rx_location.get_gcrs(rx_time)
    rx_icrs = rx_gcrs.transform_to(ac.ICRS())

    # Iterate to converge on the "bounce_time" of the apparent moon_surface, given the rx_location and rx_time.
    delta_light_travel_time = 20.0 * au.s
    bounce_time = rx_time
    approx_down_light_travel_time = 0.0 * au.s
    while np.any(np.fabs(delta_light_travel_time) > 1.0e-8 * au.s):
        moon_pos, moon_vel = ac.get_body_barycentric_posvel("moon", bounce_time)
        bc_down_dpos = moon_pos - rx_icrs.cartesian.without_differentials()
        bc_down_direction = bc_down_dpos / bc_down_dpos.norm()
        moon_surface_pos = moon_pos - MOON_RADIUS * bc_down_direction;
        distance = (moon_surface_pos - rx_icrs.cartesian.without_differentials()).norm()
        delta_light_travel_time = approx_down_light_travel_time - distance / ak.c
        approx_down_light_travel_time = distance / ak.c
        bounce_time = rx_time - approx_down_light_travel_time
    # Calculate moon_pos/vel once more with the best bounce_time    
    moon_pos, moon_vel = ac.get_body_barycentric_posvel("moon", bounce_time)
    bc_down_dpos = moon_pos - rx_icrs.cartesian.without_differentials()
    bc_down_direction = bc_down_dpos / bc_down_dpos.norm()
    # Find the moon surface sub-radar point (approximate because we're not using the lunar ellipsoid)
    moon_surface_pos = moon_pos - MOON_RADIUS * bc_down_direction;
    down_dpos = moon_surface_pos - rx_icrs.cartesian.without_differentials()
    down_range = down_dpos.norm()
    down_dvel = (moon_vel - rx_icrs.velocity)
    down_direction = down_dpos / down_range
    down_range_rate = np.dot(down_dvel.xyz.to(au.m / au.s).value, down_direction.xyz) * au.m / au.s

    print(down_dvel)
    down_ortho_rate = np.sqrt(sum(down_dvel.xyz.to(au.m/au.s).value**2) - down_range_rate.to(au.m/au.s).value**2) * au.m / au.s
    print(down_ortho_rate)

    # Iterate to converge on the "emitted_time" of the apparent tx_location, given the moon_surface_loc and bounce_time.
    delta_light_travel_time = 20.0 * au.s
    emitted_time = rx_time - approx_down_light_travel_time
    approx_up_light_travel_time = 0.0 * au.s
    while np.any(np.fabs(delta_light_travel_time) > 1.0e-8 * au.s):
        tx_gcrs = tx_location.get_gcrs(emitted_time)
        tx_icrs = tx_gcrs.transform_to(ac.ICRS())
        distance = (tx_icrs.cartesian.without_differentials() - moon_surface_pos).norm()
        delta_light_travel_time = approx_up_light_travel_time - distance / ak.c
        approx_up_light_travel_time = distance / ak.c
        emitted_time = rx_time - approx_down_light_travel_time - approx_up_light_travel_time
    # Calculate tx_icrs once more with the best emitted_time
    tx_gcrs = tx_location.get_gcrs(emitted_time)
    tx_icrs = tx_gcrs.transform_to(ac.ICRS())
    up_dpos = tx_icrs.cartesian.without_differentials() - moon_surface_pos
    up_range = up_dpos.norm()
    up_dvel = (tx_icrs.velocity - moon_vel)
    up_direction = up_dpos / up_range
    up_range_rate = np.dot(up_dvel.xyz.to(au.m / au.s).value, up_direction.xyz) * au.m / au.s

    doppler = frequency * (up_range_rate + down_range_rate) / ak.c
    delay = (up_range + down_range) / ak.c

    return doppler, delay

In [ ]:
from astroquery.jplhorizons import Horizons
# See https://ssd.jpl.nasa.gov/horizons/manual.html#observer-table for details.

def moonDopplerAndDelay_horizons(frequency, rx_time, tx_location, rx_location):
    # Observe the moon center from rx_location at rx_time.
    down_moon_center = Horizons(id='301', location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=rx_time.jd)
    down_moon_center_eph = down_moon_center.ephemerides()
    
    down_moon_center_range = down_moon_center_eph['delta'].to(au.m)[0]
    down_moon_center_range_rate = down_moon_center_eph['delta_rate'].to(au.m / au.s)[0]
    down_moon_center_light_travel_time = down_moon_center_eph['lighttime'].to(au.s)[0]  # Note: this isn't quite equal to range / c ?!?

    # Why aren't these the same? Numerical representation noise?
    #print("light times:", down_moon_center_eph['delta'].to(au.m) / ak.c, down_moon_center_eph['lighttime'].to(au.s)[0])

    # Get the sub radar point (SRP)
    bounce_lat = down_moon_center_eph['PDObsLat'].to(au.deg)[0]
    bounce_lon = down_moon_center_eph['PDObsLon'].to(au.deg)[0]
    #print("      SRP lat/lon:", bounce_lat, bounce_lon)

    # Also look up the SRP dt second later.
    dt = 0.1 * au.s
    later_down_moon_center = Horizons(id='301', location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=(rx_time + dt).jd)
    later_down_moon_center_eph = later_down_moon_center.ephemerides()
    later_bounce_lat = later_down_moon_center_eph['PDObsLat'].to(au.deg)[0]
    later_bounce_lon = later_down_moon_center_eph['PDObsLon'].to(au.deg)[0]
    #print("Later SRP lat/lon:", later_bounce_lat, later_bounce_lon)
    
    # Use the two SRPs to calculate the perpendicular rotation limb points (where the radial doppler should be maximum)
    angular_distance = 90 * au.deg
    pos_limb_lat, pos_limb_lon = calculate_delta_point(bounce_lat, bounce_lon, later_bounce_lat, later_bounce_lon, angular_distance)
    neg_limb_lat, neg_limb_lon = calculate_delta_point(bounce_lat, bounce_lon, later_bounce_lat, later_bounce_lon, -angular_distance)
    # neg limb point is on the other side of the moon...
    #print("POS LIMB lat/lon:", pos_limb_lat, pos_limb_lon)
    #print("NEG LIMB lat/lon:", neg_limb_lat, neg_limb_lon)
    
    # And query the apparent range rate of the limb points
#    approx_limb_rx_time = rx_time + MOON_RADIUS / ak.c
    approx_limb_rx_time = rx_time # HACK
    down_moon_pos_limb = Horizons(id={'body':'301', 'lon': pos_limb_lon, 'lat': pos_limb_lat, 'elevation': 0 * au.m}, location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=approx_limb_rx_time.jd)
    down_moon_pos_limb_eph = down_moon_pos_limb.ephemerides()
    down_moon_pos_limb_range_rate = down_moon_pos_limb_eph['delta_rate'].to(au.m / au.s)[0]
    #print(f"{down_moon_pos_limb_range_rate=}")
    #print(f"{down_moon_pos_limb_range_rate - down_moon_center_range_rate=}")
    down_moon_neg_limb = Horizons(id={'body':'301', 'lon': neg_limb_lon, 'lat': neg_limb_lat, 'elevation': 0 * au.m}, location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=approx_limb_rx_time.jd)
    down_moon_neg_limb_eph = down_moon_neg_limb.ephemerides()
    down_moon_neg_limb_range_rate = down_moon_neg_limb_eph['delta_rate'].to(au.m / au.s)[0]
    #print(f"{down_moon_neg_limb_range_rate=}")
    #print(f"{down_moon_neg_limb_range_rate - down_moon_center_range_rate=}")
    
    # These three should be almost exactly equal... And they aren't!?!!
    print(down_moon_pos_limb_eph['delta'].to(au.m)[0])
    print(down_moon_neg_limb_eph['delta'].to(au.m)[0])
    print(down_moon_center_range)

    # Refine the down leg by querying the apparent sub radar point (from down_moon_center).
    down_moon_srp = Horizons(id={'body':'301', 'lon': bounce_lon, 'lat': bounce_lat, 'elevation': 0 * au.m}, location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=rx_time.jd)
    down_moon_srp_eph = down_moon_srp.ephemerides()
    down_moon_srp_range = down_moon_srp_eph['delta'].to(au.m)[0]
    down_moon_srp_range_rate = down_moon_srp_eph['delta_rate'].to(au.m / au.s)[0]
    down_moon_srp_light_travel_time = down_moon_srp_eph['lighttime'].to(au.s)[0]  # Note: this isn't quite equal to range / c ?!?
    srp_bounce_time = rx_time - down_moon_srp_light_travel_time
    
    # Observing the tx_location from the moon sub radar point at bounce time
    srp_up = Horizons(id={'lon': STOCKERT_LON, 'lat': STOCKERT_LAT, 'elevation': STOCKERT_HEIGHT}, location={'body':'301', 'lon': bounce_lon, 'lat': bounce_lat, 'elevation': 0 * au.m}, epochs=srp_bounce_time.jd)
    srp_up_eph = srp_up.ephemerides()
    srp_up_range = srp_up_eph['delta'].to(au.m)[0]
    srp_up_range_rate = srp_up_eph['delta_rate'].to(au.m / au.s)[0]
    
    doppler = frequency * (srp_up_range_rate + down_moon_srp_range_rate) / ak.c
    delay = (srp_up_range + down_moon_srp_range) / ak.c

    down_moon_pos_limb_light_travel_time = down_moon_pos_limb_eph['lighttime'].to(au.s)[0]  # Note: this isn't quite equal to range / c ?!?
    pos_limb_bounce_time = rx_time - down_moon_pos_limb_light_travel_time
    down_moon_neg_limb_light_travel_time = down_moon_neg_limb_eph['lighttime'].to(au.s)[0]  # Note: this isn't quite equal to range / c ?!?
    neg_limb_bounce_time = rx_time - down_moon_neg_limb_light_travel_time
    
    pos_limb_up = Horizons(id={'lon': STOCKERT_LON, 'lat': STOCKERT_LAT, 'elevation': STOCKERT_HEIGHT}, location={'body':'301', 'lon': pos_limb_lon, 'lat': pos_limb_lat, 'elevation': 0 * au.m}, epochs=pos_limb_bounce_time.jd)
    pos_limb_up_eph = pos_limb_up.ephemerides()
    pos_limb_up_range = pos_limb_up_eph['delta'].to(au.m)[0]
    pos_limb_up_range_rate = pos_limb_up_eph['delta_rate'].to(au.m / au.s)[0]
    
    neg_limb_up = Horizons(id={'lon': STOCKERT_LON, 'lat': STOCKERT_LAT, 'elevation': STOCKERT_HEIGHT}, location={'body':'301', 'lon': neg_limb_lon, 'lat': neg_limb_lat, 'elevation': 0 * au.m}, epochs=neg_limb_bounce_time.jd)
    neg_limb_up_eph = neg_limb_up.ephemerides()
    neg_limb_up_range = neg_limb_up_eph['delta'].to(au.m)[0]
    neg_limb_up_range_rate = neg_limb_up_eph['delta_rate'].to(au.m / au.s)[0]
    
    pos_limb_doppler_delta = frequency * (pos_limb_up_range_rate + down_moon_pos_limb_range_rate) / ak.c - doppler
    neg_limb_doppler_delta = frequency * (neg_limb_up_range_rate + down_moon_neg_limb_range_rate) / ak.c - doppler
    #print('+90 limb doppler delta', pos_limb_doppler_delta)
    #print('-90 limb doppler delta', neg_limb_doppler_delta)
    min_doppler_delta = min(pos_limb_doppler_delta, neg_limb_doppler_delta)
    max_doppler_delta = max(pos_limb_doppler_delta, neg_limb_doppler_delta)
    return doppler, delay, min_doppler_delta, max_doppler_delta

In [ ]:
# Download SPICE kernels for cspyce

#! mkdir --parents spicey_kernels
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de440s.bsp
##! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/de442s.bsp
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/earth_latest_high_prec.bpc
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/moon_pa_de440_200625.bpc
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/fk/satellites/moon_de440_250416.tf
#! cd spicey_kernels; wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/generic_kernels/fk/satellites/moon_de440_250416.tf

# Generate SPICE kernels for the RX and TX sites, as specified in observatories.def, so they can be referenced. Uses the pinpoint binary.
#! wget --no-clobber https://naif.jpl.nasa.gov/pub/naif/utilities/PC_Linux_64bit/pinpoint
#! chmod u+x pinpoint
#! ./pinpoint -def observatories.defs -pck spicey_kernels/pck00011.tpc -spk spicey_kernels/observatories.bsp -fk spicey_kernels/observatories.tf

In [ ]:
import cspyce as csp
csp.tkvrsn('TOOLKIT')
SPICEY_KERNEL_DIR = "spicey_kernels"

# For guidance on high-accuracy kernels for Earth and Moon, see
# https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/Tutorials/pdf/individual_docs/23_lunar-earth_pck-fk.pdf
# For Earth, use ITRF93 frame
# For Moon, use MOON_ME frame

csp.kclear()
csp.furnsh(f"{SPICEY_KERNEL_DIR}/naif0012.tls")                # For timestamp conversions
csp.furnsh(f"{SPICEY_KERNEL_DIR}/de440s.bsp")                  # For Earth and Moon positions (use 440 for compatibility with Moon orientation below)
csp.furnsh(f"{SPICEY_KERNEL_DIR}/pck00011.tpc")                # For Earth and Moon size and shape
csp.furnsh(f"{SPICEY_KERNEL_DIR}/earth_latest_high_prec.bpc")  # For best Earth orientation
csp.furnsh(f"{SPICEY_KERNEL_DIR}/moon_pa_de440_200625.bpc")    # For best Moon orientation (needs moon_de440_220930.tf)
csp.furnsh(f"{SPICEY_KERNEL_DIR}/moon_de440_250416.tf")        # For best Moon orientation
csp.furnsh(f"{SPICEY_KERNEL_DIR}/observatories.bsp")           # For observatory positions
csp.furnsh(f"{SPICEY_KERNEL_DIR}/observatories.tf")            # For observatory frames

In [ ]:
# Cribbed from Thomas' notebook, in turn from https://github.com/daniestevez/jupyter_notebooks/blob/master/CAMRAS-EVE/Earth-Venus-Earth%20experiment%20analysis.ipynb
def moonDopplerAndDelay_surface_spice(rx_astrotime, surf_vec):
    tx_name = "STOCKERT"
    rx_name = "DWINGELOO"
    target_name = "MOON"
    target_frame = "MOON_ME"
    ref_frame = "ITRF93"
    
    tx_id = csp.bodn2c(tx_name)
    rx_id = csp.bodn2c(rx_name)
    target_id = csp.bodn2c(target_name)
    
    rx_time = csp.str2et(rx_astrotime.utc.value)
    target_radius = csp.bodvrd("MOON", "RADII", 3)[1][0]
    surf_vec = np.array(surf_vec)
    surf_vec = target_radius * csp.unorm(surf_vec)[0]
    ez_r, lt_r = csp.spkcpt(surf_vec, "MOON", "MOON_ME", rx_time, "J2000", "OBSERVER", "CN", "DWINGELOO")
    ez_d, lt_d = csp.spkcpo("DWINGELOO", rx_time - lt_r, "J2000", "OBSERVER", "CN", surf_vec, "MOON", "MOON_ME")  # Monostatic: RX by Dwingeloo
    #ez_d, lt_d = sp.spkcpo("STOCKERT", rx_time - lt_r, "J2000", "OBSERVER", "CN", surf_vec, "MOON", "MOON_ME")  # Bistatic: RX by Stockert
    
    dlt_r = csp.dvnorm(ez_r) / csp.clight()
    ez_d[3:] *= 1 - dlt_r  # See https://destevez.net/2025/04/analysis-of-the-camras-venus-radar-experiment/
    dlt_d = csp.dvnorm(ez_d) / csp.clight()
    return dlt_r + dlt_d, lt_r + lt_d

def moonDopplerAndDelay_spice(frequency, rx_astrotime, tx_location, rx_location):
    ## HACK: tx_location and rx_location are not used...
    rx_time = csp.str2et(rx_astrotime.utc.value)
    tx_name = "DWINGELOO"
    rx_name = "STOCKERT"
    target_name = "MOON"
    target_frame = "MOON_ME"
    ref_frame = "ITRF93"
    abcorr = "CN"  # Aberration correction method.
    
    #tx_id = sp.bodn2c(tx_name)
    #rx_id = sp.bodn2c(rx_name)
    #target_id = sp.bodn2c(target_name)
    
    #ref_frame = "J2000"

    
    # Heavily inspired from https://github.com/daniestevez/jupyter_notebooks/blob/master/CAMRAS-EVE/Earth-Venus-Earth%20experiment%20analysis.ipynb
    
    #down_moon_center = Horizons(id='301', location={'lon': DWINGELOO_LON, 'lat': DWINGELOO_LAT, 'elevation': DWINGELOO_HEIGHT}, epochs=rx_start_astrotime.jd)
    #down_moon_center_eph = down_moon_center.ephemerides()
    #display(down_moon_center_eph)
    #print(f"jpl range        {down_moon_center_eph['delta'].to(au.km)[0]}")
    #print(f"jpl range rate   {down_moon_center_eph['delta_rate'][0]}")
    #print(f"jpl lighttime    {down_moon_center_eph['lighttime'].to(au.s)[0]}")
    
    ## Simple query to the Moon center
    ## Working backwards, observe target_name from rx_name at rx_time,
    ## compute the position and velocity of the target relative to the observer,
    ## corrected for light-time ('CN').
    #posvel_r, lt_r = sp.spkezr(target_name, rx_time, ref_frame, 'CN', rx_name)
    ##print(f"spice range      {sp.vnorm(posvel_r)}")
    ##print(f"spice range rate {sp.dvnorm(posvel_r)}")
    ##print(f"spice lighttime  {lt_r}")
    
    ## Then observe tx_name from target_name at bounce_time.
    #posvel_d, lt_d = sp.spkezr(tx_name, rx_time - lt_r, ref_frame, 'CN', target_name)
    #dlt_r = sp.dvnorm(posvel_r) / sp.clight()  # Calculate the range rate
    #posvel_d[3:] *= 1 - dlt_r   # See https://destevez.net/2025/04/analysis-of-the-camras-venus-radar-experiment/
    #dlt_d = sp.dvnorm(posvel_d) / sp.clight()  # Calculate the range rate
    #dlt = dlt_r + dlt_d
    #print(dlt * frequency)
    
    ## Seems cleaner to use spkltc... but unfortunately it only supports bodies (so we can't use it for SRP):
    #stobs = sp.spkssb(rx_id, rx_time, ref_frame)
    #posvel_r, lt_r, dlt_r = sp.spkltc(target_id, rx_time, ref_frame, 'CN', stobs)
    #posvel_d, lt_d, dlt_d = sp.spkltc(tx_id, rx_time - lt_r, ref_frame, 'CN', stobs + posvel_r)
    #dlt = dlt_r + dlt_d
    #print(dlt * frequency)
    
    # Use the sub-radar point.
    # Compute the rectangular coordinates (in the target_frame) of the sub rx observatory point, corrected for light time and stellar aberration.
    srp, t_ref, _ = csp.subpnt('INTERCEPT/ELLIPSOID', target_name, rx_time, target_frame, abcorr, rx_name)
    # Compute the state (in the ref_frame) of the sub rx observatory point (in the target_frame), relative to the rx observatory.
    srp_r, lt_r = csp.spkcpt(srp, target_name, target_frame, rx_time, ref_frame, 'OBSERVER', abcorr, rx_name)
    # Compute the state (in the target_frame) of the tx observatory (in the ref_frame) relative to a fictional observer at the sub rx point, where the observer has constant position in the target frame.
    tx_d, lt_d = csp.spkcpo(tx_name, t_ref, ref_frame, 'OBSERVER', abcorr, srp, target_name, target_frame)
    # Compute the radial velocity, and divide by c to get the doppler factor
    dlt_r = csp.dvnorm(srp_r) / sp.clight()
    # Adjust the tx observer velocity: see https://destevez.net/2025/04/analysis-of-the-camras-venus-radar-experiment/
    tx_d[3:] *= 1 - dlt_r  
    # Compute the radial velocity, and divide by c to get the doppler factor
    dlt_d = csp.dvnorm(tx_d) / sp.clight()
    dlt = dlt_r + dlt_d
    return dlt * frequency, (lt_r + lt_d) * au.s

In [ ]:
# Cribbed from Thomas' notebook, in turn from https://github.com/daniestevez/jupyter_notebooks/blob/master/CAMRAS-EVE/Earth-Venus-Earth%20experiment%20analysis.ipynb
def radar_surface_dlt(rx_astrotime, vec):
    ref_frame = "ITRF93"  # "J2000"
    ab_cor = "LT" #"CN"
    rx_time = csp.str2et(rx_astrotime.utc.value)
    # Scale the surface vector so it lies on the surface of the moon's triaxial ellipsoid.
    moon_radii = csp.bodvrd("MOON", "RADII")
    surf_vec = csp.edpnt_vector(vec, moon_radii[0], moon_radii[1], moon_radii[2])
    # Compute the state (in ref_frame) of the surface point (in MOON_ME frame) as seen by the rx observatory at time rx_time, corrected for light time and stellar aberration.
    sr_from_rx, lt_rx = csp.spkcpt_vector(surf_vec, "MOON", "MOON_ME", rx_time, ref_frame, "OBSERVER", ab_cor, "DWINGELOO")
    # Compute the state of the tx observatory (in ref_frame) as seen by a fictional observer at the surface point (in MOON_ME frame) at time rx_time - lt_rx, corrected for light time and stellar aberration.
    tx_from_sr, lt_tx = csp.spkcpo_vector("STOCKERT", rx_time - lt_rx, ref_frame, "OBSERVER", ab_cor, surf_vec, "MOON", "MOON_ME")  # Bistatic: RX by Stockert
#    tx_from_sr[3:] *= 1 - dlt_r  # See https://destevez.net/2025/04/analysis-of-the-camras-venus-radar-experiment/
    return (csp.dvnorm_vector(sr_from_rx) + csp.dvnorm_vector(tx_from_sr)) / csp.clight(), lt_rx + lt_tx
    
if 1: # Debug the computed doppler/delay. 
    NSIDE = 20
    NPIX = hp.nside2npix(NSIDE)
    healpix_resol_deg = hp.nside2resol(NSIDE, arcmin=True) / 60
    print(f"HEALPix {NSIDE=} {NPIX=} {healpix_resol_deg=:.2f}")

    v = np.array(hp.pix2vec(NSIDE, np.arange(NPIX))).T
    print(f"{v.shape=}")
    dlt, lt = radar_surface_dlt(rx_start_astrotime, v)
    
    hp.orthview(dlt, flip='geo', title='Moon Surface Doppler', unit='Hz')
    hp.graticule()
    
    hp.orthview(lt, flip='geo', title='Moon Surface Delay', unit='s')
    hp.graticule()
    
if 0:    
    # Compute the state (in the MOON_ME frame) of a target observatory relative to a fictional observer on the moon, corrected for light time and steellar aberration.
    srp, _ = sp.spkpos("STOCKERT", sp.str2et(rx_start_astrotime.utc.value), "MOON_ME", "CN", "MOON")
    #srp, _ = sp.spkpos("DWINGELOO", sp.str2et(rx_start_astrotime.utc.value), "MOON_ME", "CN", "MOON")
    # Convert the rectangular coordinates to [lon_deg, lat_deg]
    srp_deg = np.rad2deg(sp.reclat(srp)[1:])
    print(f"{srp_deg=}")

#    ## Look up the "start" doppler from the dlt_surface
#    #srp_ipix = hp.ang2pix(nside=NSIDE, theta=srp_deg[0], phi=srp_deg[1], lonlat=True)
#    #doppler_start = dlt_surface[srp_ipix] * frequency
#    
#    # Compute "start" doppler using radar_surface_dlt 
#    doppler_start, delay_start = radar_surface_dlt(rx_start_astrotime, srp) * frequency
#
    # Compute "start" doppler using standalone function
    doppler_start, delay_start = moonDopplerAndDelay_spice(frequency, rx_start_astrotime, tx_location, rx_location)
    
    print(f"{doppler_start=}")
    doppler_surface_offset = doppler_start - dlt_surface * frequency
    
#    print("doppler_surface_offset at SRP", doppler_surface_offset[srp_ipix])
#    #doppler_surface_offset[ipix] = 20 * au.Hz  # Overwrite at the SRP location
#
#    ## Paint the computed min/max doppler regions
#    #ivec = hp.ang2vec(theta=86, phi=-25, lonlat=True)
#    #ipix = hp.query_disc(nside=NSIDE, vec=ivec, radius=np.radians(10))
#    #print(ipix)
#    #doppler_surface_offset[ipix] = doppler_surface_offset.max()
#    #
#    #ivec = hp.ang2vec(theta=-86, phi=25, lonlat=True)
#    #ipix = hp.query_disc(nside=NSIDE, vec=ivec, radius=np.radians(10))
#    #print(ipix)
#    #doppler_surface_offset[ipix] = doppler_surface_offset.min()
#    
    hp.orthview(doppler_surface_offset, flip='geo', rot=srp_deg, half_sky=True,
                title='Moon Surface Doppler offset', unit='Hz')
    hp.graticule()
    
    doppler_surface_offset_min = np.nanmin(doppler_surface_offset, axis=0)
    doppler_surface_offset_max = np.nanmax(doppler_surface_offset, axis=0)
    print("min, max, span:", doppler_surface_offset_min, doppler_surface_offset_max, doppler_surface_offset_max + doppler_surface_offset_min)

In [ ]:
if _interactive:  # Compute the doppler angle using the delta srp method
    srp0, _ = sp.spkpos("STOCKERT", sp.str2et(rx_start_astrotime.utc.value), "MOON_ME", "CN+S", "MOON")
    srp0_rad = np.array(sp.reclat(srp0)[1:])
    srp1, _ = sp.spkpos("STOCKERT", sp.str2et((rx_start_astrotime.utc + 0.1 * au.s).value), "MOON_ME", "CN+S", "MOON")
    srp1_rad = np.array(sp.reclat(srp1)[1:])
    dsrp = srp0_rad - srp1_rad
    doppler_angle = np.arctan2(-dsrp[1], -dsrp[0] * np.cos(srp0[1])) * au.radian
    print(f"{doppler_angle=}")

In [ ]:
if _interactive: # Calculate predicted doppler center frequency
    #doppler_start, delay_start = moonDopplerAndDelay_astropy(rx_start_astrotime, tx_location, rx_location)
    #doppler_start, delay_start, min_doppler_delta_start, max_doppler_delta_start = moonDopplerAndDelay_horizons(frequency, rx_start_astrotime, tx_location, rx_location)
    doppler_start, delay_start = moonDopplerAndDelay_spice(frequency, rx_start_astrotime, tx_location, rx_location)
    print(f"{doppler_start=} {delay_start=}")
#    print(f"{doppler_start=} {delay_start=} {min_doppler_delta_start=} {max_doppler_delta_start=} ")
#    print("asymmetry in doppler", min_doppler_delta_start + max_doppler_delta_start)
    
    rx_duration = len(rx_samples) / sample_rate
    print(f"{rx_duration=}")
    rx_end_astrotime = rx_start_astrotime + rx_duration
    
#    #doppler_end, delay_end = moonDopplerAndDelay_astropy(rx_end_astrotime, tx_location, rx_location)
#    doppler_end, delay_end = moonDopplerAndDelay_horizons(rx_end_astrotime, tx_location, rx_location)
    doppler_end, delay_end = moonDopplerAndDelay_spice(frequency, rx_end_astrotime, tx_location, rx_location)
    print(f"{doppler_end=} {delay_end=}")
    
    # Simple doppler_rate model, linear from start to end. Generally good enough for 30s windows?
    doppler_rate = (doppler_end - doppler_start) / rx_duration
    print(f"{doppler_rate=}")

In [ ]:
if 0:  # HACK: manually override doppler delay parameters
    print("HACK HACK HACK")
    
    # These are from Thomas' notebook.
    # FOR: rx-2025-03-11/stockert_eme_2025_03_11_19_36_21_1297.500MHz_1.00Msps_ci16_le.chanXXX.sigmf-meta
    # ... but they're not perfect (see peak correlation and image symmetry checks)
    #doppler_start = -1046.669 * au.Hz
    #doppler_rate = 0.13120475496564593 * au.Hz / au.s 
    #delay_start = 2.592436 * au.s 

    ## MANUAL BEST FIT (first doppler_start using delay-doppler image left-right symmetry), then delay_start using the correlation spike.
    # FOR: rx-2025-03-11/stockert_eme_2025_03_11_19_36_21_1297.500MHz_1.00Msps_ci16_le.chanXXX.sigmf-meta
    doppler_start = -1046.605 * au.Hz
    doppler_rate = 0.1310 * au.Hz / au.s 
    delay_start = 2.59245 * au.s

    print(f"{doppler_start=}")
    print(f"{delay_start=}")
    print(f"{doppler_rate=}")

In [ ]:
def calculateDelayWindowIndices(sample_rate, delay_start, len_rx_samples, len_tx_samples):
    moon_delay_depth = MOON_RADIUS / ak.c * 2  # round-trip delay depth

    # Padded
    #t_start = delay_start - 0.001 * au.s
    #t_end = delay_start + moon_delay_depth + 0.001 * au.s
    
    ## Tight 
    #t_start = delay_start
    #t_end = delay_start + moon_delay_depth - 0.001 * au.s
    
    # Truncated
    t_start = delay_start
    t_end = delay_start + moon_delay_depth - 0.001 * au.s
    
    cor_lags = scipy.signal.correlation_lags(len_rx_samples, len_tx_samples, mode="valid") / sample_rate
    cor_i_start = np.argwhere(cor_lags > t_start)[0][0]
    cor_i_end = np.argwhere(cor_lags > t_end)[0][0]
    
    return cor_lags, cor_i_start, cor_i_end

if _interactive:  # Calculate the delay window of interest
    cor_lags, cor_i_start, cor_i_end = calculateDelayWindowIndices(sample_rate, TX_START + delay_start, len(rx_samples), len(tx_samples))

In [ ]:
if 0: # Search for best doppler using peak correlation strength
    # If the current hypothesis is that the empirical doppler error is primarily caused by tx and rx frequency errors,
    # then we are searching for the sum of the frequency errors. If these errors don't change much over the course
    # of an observation (a big 'if'), then I think we can assume the doppler_rate is pretty good, don't have to search over it.
    
    # Do everything in GPU memory, if it fits!
    cupy_f_shifts = np.linspace(doppler_start.to(au.Hz).value - 0.2, doppler_start.to(au.Hz).value + 0.2, 1000)
    cupy_tx_samples = cupy.asarray(tx_samples, dtype=cupy.complex64)
    cupy_rx_samples = cupy.asarray(rx_samples, dtype=cupy.complex64)
    best_peak_doppler = doppler_start
    best_peak = 0.0
    
    # Cache computations/allocations outside the for loop.
    cupy_conj_fft_tx_samples = cupy.conj(cupy.fft.fft(cupy.pad(cupy_tx_samples, (0, len(rx_samples) - len(tx_samples)))))
    #cupy_rx_range = -1j * 2 * cupy.pi / sample_rate.value * cupy.arange(len(rx_samples))
    cupy_rx_t_s = cupy.arange(len(rx_samples)) / sample_rate.value
    
    #for i in tqdm(range(len(cupy_f_shifts))):
    for i in range(len(cupy_f_shifts)):
        #cupy_rx_samples_shifted = cupy_rx_samples * cupy.exp(cupy_f_shifts[i] * cupy_rx_range)
        cupy_rx_samples_shifted = cupy_rx_samples * cupy.exp(-1j * 2 * np.pi * (-cupy_f_shifts[i] * cupy_rx_t_s + (-doppler_rate.value * cupy_rx_t_s**2) / 2)).T
        #cupy_rx_samples_shifted = cupy_rx_samples * cupy.exp(-1j * 2 * cupy.pi / sample_rate.value * cupy.arange(len(rx_samples))
        cupy_cor_shifted = cupy.fft.ifft(cupy.fft.fft(cupy_rx_samples_shifted) * cupy_conj_fft_tx_samples)
        #cor_max = cupy.max(cupy.abs(cupy_cor_shifted))
        #cor_max = cupy.max(cupy.abs(cupy_cor_shifted[cor_i_start:(cor_i_start+2000)]))
        cor_max = cupy.max(cupy.abs(cupy_cor_shifted[cor_i_start:cor_i_end]))
        if cor_max > best_peak:
            best_peak = cor_max
            best_peak_doppler = cupy_f_shifts[i]
            #print(i, best_peak, best_peak_doppler)

    print(f"{best_peak_doppler=}")
    print(f"Calculated vs search doppler delta: {doppler_start - best_peak_doppler * au.Hz}")

    doppler_start = best_peak_doppler * au.Hz

    if 1: # Debug
        phi_Hz = (-best_peak_doppler * cupy_rx_t_s) + (-doppler_rate.value * cupy_rx_t_s**2) / 2
        cupy_rx_samples_shifted = cupy_rx_samples * cupy.exp(-1j * 2 * np.pi * phi_Hz).T
        cupy_cor_shifted = cupy.fft.ifft(cupy.fft.fft(cupy_rx_samples_shifted) * cupy_conj_fft_tx_samples)
        pl.figure()
        pl.title("RX-compensated/TX correlation (delay window)")
        pl.plot(cor_lags[cor_i_start:cor_i_end] / sample_rate, cupy.abs(cupy_cor_shifted[cor_i_start:cor_i_end]).get(), label="signal")
        pl.gca().axvline(TX_START.value + delay_start.value, linestyle="--", color="grey", label="expected based on Moon distance",)
        #pl.plot(cupy.abs(cupy_cor_shifted[cor_i_start:(cor_i_start+2000)]).get())
        rx_samples_compensated = cupy_rx_samples_shifted.get()

In [ ]:
if _interactive: # Compensate for time-varying doppler (sum of dopplers from transmitter -> moon, and moon -> receiver)
    # Slow on CPU, but just doing it once...
    t_s = np.arange(len(rx_samples)) / sample_rate
    phi_Hz = (-doppler_start * t_s) + (-doppler_rate * t_s**2) / 2  # Instantaneous phase.
    rx_samples_compensated = rx_samples * np.exp(-1j * 2 * np.pi * phi_Hz.value).T

In [ ]:
if 0: # Debug: compensated rx spectrogram
    %matplotlib widget
    pl.figure()
    nfft = nperseg = 2**8
    Sxx_rx, f_rx, t_rx, image_rx = pl.specgram(rx_samples_compensated, Fs=sample_rate, NFFT=nfft, clim=[-110, -80])
    pl.grid()

In [ ]:
if 0:  # Debug: check tx signal autocorrelation
    cor = scipy.signal.correlate(tx_samples, tx_samples, mode="full")
    cor_lags = scipy.signal.correlation_lags(len(tx_samples), len(tx_samples), mode="full") / sample_rate
    pl.figure()
    pl.plot(cor_lags, np.abs(cor), label="signal")
    pl.title("TX auto-correlation")
    #pl.title("Zadoff Chu auto-correlation")
    #pl.title("BPSK auto-correlation")

In [ ]:
if 0:  # Debug: check calculated delay and doppler by looking at the correlation of the rx_compensated and tx signals.
    #%matplotlib widget
    # Slow on CPU, but just doing it once...
    #cor = scipy.signal.correlate(rx_samples_compensated, tx_samples, mode="valid")
    cor = scipy.signal.correlate(rx_samples, tx_samples, mode="valid")
    cor_lags = scipy.signal.correlation_lags(len(rx_samples), len(tx_samples), mode="valid") / sample_rate
    pl.subplot(211)
    pl.title("RX-compensated/TX correlation (full)")
    pl.plot(cor_lags, np.abs(cor), label="signal")
    pl.gca().axvline(TX_START.value + delay_start.value, linestyle="--", color="grey", label="expected based on Moon distance",)
    pl.legend()
    pl.subplot(212)
    pl.title("RX-compensated/TX correlation (delay window)")
    #pl.plot(cor_lags, np.abs(cor), label="signal")
    #pl.gca().set_xlim([TX_START.value + delay_start.value - 0.001, TX_START.value + delay_start.value + 0.001])
    pl.plot(cor_lags[cor_i_start:cor_i_end], np.abs(cor[cor_i_start:cor_i_end]), label="signal")
    pl.gca().axvline(TX_START.value + delay_start.value, linestyle="--", color="grey", label="expected based on Moon distance",)

In [ ]:
# CUDA implementation: about 50 it/s on an nvidia rtx 3080
def correlateSignalCUDA(tx_samples, rx_samples, cor_i_start, cor_i_end, f_shifts):
    # Do everything in GPU memory, if it fits!
    tx_samples = cupy.asarray(tx_samples, dtype=cupy.complex64)
    rx_samples = cupy.asarray(rx_samples, dtype=cupy.complex64)
    A = cupy.asarray(np.zeros((len(f_shifts), cor_i_end - cor_i_start)))
    # Cache computations/allocations outside the for loop.
    fft_rx_samples = cupy.fft.fft(rx_samples)
    tx_samples_shifted = cupy.zeros_like(rx_samples)
    tx_range = -1j * 2 * cupy.pi / sample_rate.value * cupy.arange(len(tx_samples)) 
    for i in tqdm(range(len(f_shifts))):
        tx_samples_shifted[:len(tx_samples)] = tx_samples * cupy.exp(f_shifts[i] * tx_range)
        cor_shifted = cupy.fft.ifft(fft_rx_samples * cupy.conj(cupy.fft.fft(tx_samples_shifted)))
        A[i] = cupy.abs(cor_shifted[cor_i_start:cor_i_end])
    # Single copy back to CPU at the end.
    return cupy.asnumpy(A)

In [ ]:
if _interactive:  # Compute the doppler-delay image.
    f_shifts = np.linspace(dlt_surface.min() * frequency.value, dlt_surface.max() * frequency.value, 1000) - doppler_start.value
    A = correlateSignalCUDA(tx_samples, rx_samples_compensated, cor_i_start, cor_i_end, f_shifts)
    log_A = np.log(A)

In [ ]:
if _interactive: # Show the initial doppler-delay image
    pl.figure(figsize=(16,16))
    pl.imshow(log_A.T, 
              extent=[
                  f_shifts[0],
                  f_shifts[-1],
                  cor_lags[cor_i_start].value,
                  cor_lags[cor_i_end].value,
              ],
              aspect='auto',
              vmin=log_A.max() * 0.4,
              vmax=log_A.max() * 0.8, 
             )
    os.makedirs(f"{DATA_ROOT}/DOPPLER_TRIAGE", exist_ok=True)
    pl.savefig(f"{DATA_ROOT}/DOPPLER_TRIAGE/{rx_chan0_sigmf_filename.split('/')[-1]}_doppler.png")

In [ ]:
# Accumulate the Doppler/Delay data into a Spherical-to-Planar projection
    
# 0 lon in the center of the map, lon in [-pi, pi), lat in [-pi, pi)
def setLLV(G, Gc, lon, lat, v):
    r = ((lat + np.pi / 2) / np.pi * G.shape[0]).astype('i')
    c = ((lon + np.pi / 2) / np.pi * G.shape[1]).astype('i')
    G[r, c] = v
    Gc[r, c] = 1

def addLLV(G, Gc, lon, lat, v):
    r = ((lat + np.pi / 2) / np.pi * G.shape[0]).astype('i')
    c = ((lon + np.pi / 2) / np.pi * G.shape[1]).astype('i')
    G[r, c] += v
    Gc[r, c] += 1

def dopplerDelayToSphericalProjection(DD, G, Gc,
                                      baud,
                                      srp_lon,
                                      srp_lat,
                                      doppler_angle):
    # Omit "degraded data" regions due to
    # - very bright returns near the SRP
    # - high N/S ambiguity near the radar equator (first few degrees of latitude), and
    # - grazing angle (last few degrees of latitude and longitude)
    _span = 70 # 80 HACK
    dlon = np.linspace(-_span / 180 * np.pi, _span / 180 * np.pi, 4000)
    dlat = np.linspace(-_span / 180 * np.pi, _span / 180 * np.pi, 8000)
    dlon_mesh, dlat_mesh = np.meshgrid(dlon, dlat)
    mesh_good = (np.sqrt(dlat_mesh**2 + dlon_mesh**2) > (7 / 180 * np.pi)) & (np.abs(dlat_mesh) > (5 / 180 * np.pi))
    #mesh_good = np.sqrt(dlat_mesh**2 + dlon_mesh**2) > (7 / 180 * np.pi)
    dlon_mesh = dlon_mesh[mesh_good]
    dlat_mesh = dlat_mesh[mesh_good]

    c2 = DD.shape[1] / 2
    cs = c2 * np.cos(dlat_mesh) * np.sin(dlon_mesh) + c2
    _radius_km = 1737.4  # MOON_RADIUS
    _row_dist_km = 299792.46 * baud / 2  # km  Note: row is *round-trip-time* (double distance)
    rs = (1 - np.cos(dlon_mesh) * np.cos(dlat_mesh)) * (_radius_km / _row_dist_km)
    rs = rs.astype(np.int16)
    cs = cs.astype(np.int16)
    valid = (cs >= 0) & (cs < DD.shape[1]) & (rs >= 0) & (rs < DD.shape[0])   
    
    if 0: # No SRP transform: as if the SRP were (0, 0)
        print("NO SRP TRANSFORM!")
        addLLV(G, Gc, dlon_mesh[valid], dlat_mesh[valid], DD[rs[valid], cs[valid]])

    if 1: # Transformed to the SRP and rotated by the "doppler angle"
        # Convert to unit spheroid cartesian coordinates.
        cdlat_mesh = np.cos(dlat_mesh)
        X = np.matrix((cdlat_mesh * np.cos(dlon_mesh),
                       cdlat_mesh * np.sin(dlon_mesh),
                       np.sin(dlat_mesh)))

        # The S matrix rotates the coordinate system to center the SRP.
        clon = np.cos(srp_lon)
        clat = np.cos(srp_lat)
        slon = np.sin(srp_lon)
        slat = np.sin(srp_lat)
        S = np.matrix((( clon * clat,  slon * clat, slat),
                       (       -slon,         clon,    0),
                       (-clon * slat, -slon * slat, clat))) 

        ## The D matrix rotates the coordinates about the x axis by the apparent doppler angle:
        cnp = np.cos(doppler_angle.to(au.radian))
        snp = np.sin(doppler_angle.to(au.radian))
        D = np.matrix(((1,   0,    0),
                       (0, cnp, -snp),
                       (0, snp,  cnp))) 

        X = (S.T * D) * X  # HACK really ought to be D.T, but I think I need to flip apparentRotationAngleDelta_poliastro
 
        # Convert back to lat/lon
        dlat_mesh = np.arcsin(X[2].A)[0]
        dlon_mesh = np.arctan2(X[1].A, X[0].A)[0]
        addLLV(G, Gc, dlon_mesh[valid], dlat_mesh[valid], DD[rs[valid], cs[valid]])
    

if 0:
    G = np.zeros((2000, 2000), dtype=np.float32) # TODO: use a smaller data rep?
    Gc = np.zeros(G.shape, dtype=np.int16)
    #setLLV(G, Gc, np.linspace(-90, 89.9, 10000) / 180 * np.pi, np.linspace(0, 20, 10000) / 180 * np.pi, 1)
    #setLLV(G, Gc, np.linspace(-90, 89.9, 10000) / 180 * np.pi, np.linspace(0, 20, 10000) / 180 * np.pi + 0.1, 1)
    dopplerDelayToSphericalProjection(log_A.T, G, Gc,
                                      1.0 / sample_rate.value,
                                      srp_deg[0] / 180 * np.pi, # srp_lon,
                                      srp_deg[1] / 180 * np.pi, # srp_lat,
                                      doppler_angle,
    )
    #np.divide(G, Gc, out=G, where=Gc>0)  # Divide in place to save memory
    #os.makedirs(f"{DATA_ROOT}/GLOBAL_TRIAGE", exist_ok=True)
    #pl.imsave(f"{DATA_ROOT}/GLOBAL_TRIAGE/{rx_chan0_sigmf_filename.split('/')[-1]}_global.png", np.flipud(G))

    if 1:
        pl.figure(figsize=(20, 10))
        pl.axis('off')
        pl.imshow(G, cmap='gray', origin='lower')

In [ ]:
if 1:  # Debug: visualize the predicted doppler/delay on the observed doppler delay image.
    B = log_A.copy()
    #B = np.zeros(log_A.shape) + 1
    #doppler_values = f_shifts * au.Hz + doppler_start - 0.10 * au.Hz  # for the 03_11 HACK HACK HACK
    doppler_values = f_shifts * au.Hz + doppler_start  # No adjustment needed for the 06_21 data!
    delay_values = cor_lags[cor_i_start:cor_i_end]
    DOPPLER_PAD = 1
    DELAY_PAD = 2
    VALUE = log_A.max()
    #VALUE = 0
    
    for i in range(len(dlt_list)):
        dlt = dlt_list[i]
        lt = lt_list[i]
        doppler_index = np.argmin((doppler_values - dlt * frequency)**2)
        delay_index = np.argmin((delay_values - ((lt + 0.00005) * au.s + TX_START))**2)  # HACK
        B[(doppler_index-DOPPLER_PAD):(doppler_index+DOPPLER_PAD+1), (delay_index-DELAY_PAD):(delay_index+DELAY_PAD+1)] = VALUE
    
    ## Vectorized version -- slower (!?) and *much* more memory intensive!
    #doppler_indices = np.argmin((doppler_values - (dlt_surface * frequency)[:, np.newaxis])**2, axis=-1)
    #delay_indices = np.argmin((delay_values - (lt_surface * au.s + TX_START)[:, np.newaxis])**2, axis=-1)
    #B[doppler_indices, delay_indices] = VALUE
    
    pl.figure(figsize=(16,16))
    pl.imshow(B.T, #B.T * log_A.T, 
              extent=[
                  f_shifts[0],
                  f_shifts[-1],
                  cor_lags[cor_i_start].value,
                  cor_lags[cor_i_end].value,
              ],
              aspect='auto',
              interpolation='none',
              vmin=log_A.max() * 0.4,
              #vmax=log_A.max() * 0.8, 
             )
    
    #os.makedirs(f"{DATA_ROOT}/PREDICT_TRIAGE", exist_ok=True)
    #pl.savefig(f"{DATA_ROOT}/PREDICT_TRIAGE/{rx_chan0_sigmf_filename.split('/')[-1]}_doppler_grid.png")

In [ ]:
if 0: # Debug the central doppler frequency by flipping horizontally and summing.
    #%matplotlib widget
    pl.figure(figsize=(16,16))
    pl.imshow(log_A.T + log_A.T[:, ::-1], aspect='auto')  
    pl.gca().axvline(log_A.shape[0] / 2.0, linestyle="--",color="grey")

In [ ]:
assert(not _interactive)

#DATA_PREFIX = f"{DATA_ROOT}sdr-eme/rx-2025-03-11/"
DATA_PREFIX = f"{DATA_ROOT}sdr-eme/rx-2025-06-21/"

if not _interactive: # Scan the available files (Stockert RX only), collect the metadata
    rx_info_map = {}
    for filename in os.listdir(DATA_PREFIX):
        if not filename.startswith('stockert'): continue
        if '_1970_' in filename: continue  # Filter out weird datetime!
        if not filename.endswith('meta'): continue
        sigmf_file = sigmf.sigmffile.fromfile(DATA_PREFIX + filename, skip_checksum=True)
        rx_info_map[filename] = sigmf_file.get_global_info()
        
if 0: # Example query across the available files...
    for rx_filename, rx_info in rx_info_map.items():
        if not 'chan0' in rx_filename: continue
        desc = rx_info["core:description"]
        if not '30sec' in desc: continue
        if not 'zadoff-chu' in desc: continue
        #if not 'bpsk' in desc: continue
        print(rx_filename, rx_info["core:description"])

In [ ]:
def toHEALPixSurface(NSIDE, log_A, rx_start_astrotime):
    doppler_values = f_shifts * au.Hz + doppler_start
    delay_values = cor_lags[cor_i_start:cor_i_end]
    NPIX = hp.nside2npix(NSIDE)
    val_surface = np.zeros(NPIX)
    for n in range(NPIX):
        v = hp.pix2vec(NSIDE, n)
        # Crudely filter out the far hemisphere.
        if v[0] < 0: continue
        dlt, lt = radar_surface_dlt(rx_start_astrotime, v)
        doppler_index = np.argmin((doppler_values - dlt * frequency)**2)
        delay_index = np.argmin((delay_values - ((lt + 0.00005) * au.s + TX_START))**2)
        val_surface[n] = log_A[doppler_index, delay_index]
    return val_surface

In [ ]:
# BATCH MODE
assert(not _interactive)

G = np.zeros((2000, 2000), dtype=np.float64)
Gc = np.zeros(G.shape, dtype=np.int32)

for rx_filename, rx_info in rx_info_map.items():
    #if not 'chan0' in rx_filename: continue
    #if not 'chan1' in rx_filename: continue
    desc = rx_info["core:description"]
    if 'cw-' in desc: continue  # Common
    if 'pulsed' in desc: continue  # Common
    #if not '30sec' in desc: continue
    #if not '60sec' in desc: continue
    if not (('60sec' in desc) or ('30sec' in desc)): continue   # For the 06_21 dataset 
    #if not '50000-' in desc: continue   # For the 03_11 dataset
    if not 'zadoff-chu' in desc: continue # Common
    #if not 'bpsk' in desc: continue

    if not (rx_filename in [
        "stockert_eme_2025_06_21_08_48_35_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_09_06_52_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_09_46_51_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_10_01_49_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_10_45_50_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_10_26_29_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_11_27_05_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        #"stockert_eme_2025_06_21_11_18_31_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        "stockert_eme_2025_06_21_11_24_12_1299.500MHz_0.25Msps_ci16_le.chan0.sigmf-meta",
        # chan1
        "stockert_eme_2025_06_21_08_48_35_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_09_06_52_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_09_46_51_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_10_01_49_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_10_45_50_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_10_26_29_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_11_27_05_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        #"stockert_eme_2025_06_21_11_18_31_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
        "stockert_eme_2025_06_21_11_24_12_1299.500MHz_0.25Msps_ci16_le.chan1.sigmf-meta",
    ]):
        continue

    print(rx_filename)

    rx_samples, tx_samples, sample_rate, frequency, rx_start_astrotime = loadRxTxFiles(DATA_PREFIX + rx_filename)
    rx_duration = len(rx_samples) / sample_rate
    tx_duration = len(tx_samples) / sample_rate
    if rx_duration < 20 * au.s:
        print(f"SKIPPING DUE TO SHORT DURATION {rx_duration=} {tx_duration=}")
        continue
    
    doppler_start, delay_start = moonDopplerAndDelay_spice(frequency, rx_start_astrotime, tx_location, rx_location)
    rx_end_astrotime = rx_start_astrotime + rx_duration
    doppler_end, delay_end = moonDopplerAndDelay_spice(frequency, rx_end_astrotime, tx_location, rx_location)
    
    # Simple doppler_rate model, linear from start to end. Generally good enough for a few minutes...
    doppler_rate = (doppler_end - doppler_start) / rx_duration
    #print(f"{doppler_rate=}")

    # Compensate the rx_samples for the time-varying doppler 
    t_s = np.arange(len(rx_samples)) / sample_rate
    phi_Hz = (-doppler_start * t_s) + (-doppler_rate * t_s**2) / 2
    rx_samples_compensated = rx_samples * np.exp(-1j * 2 * np.pi * phi_Hz.value).T
    min_delay = min(delay_start, delay_end)
    cor_lags, cor_i_start, cor_i_end = calculateDelayWindowIndices(sample_rate, TX_START + min_delay, len(rx_samples), len(tx_samples))

    # Calculate the range of doppler shifts.
    rx_start_csptime = csp.str2et(rx_start_astrotime.utc.value)
    target_epoch, observer_pos, v_term = csp.edterm("UMBRAL", "DWINGELOO", "MOON", rx_start_csptime, "MOON_ME", "CN", "STOCKERT", 1000)  # Use Dwingeloo telescope as the light source!
    dlt_term, lt_term = radar_surface_dlt(rx_start_astrotime, v_term)

    dlt_shifts = np.linspace(dlt_term.min(), dlt_term.max(), 3000)
    f_shifts = dlt_shifts * frequency.value - doppler_start.value
#    A = correlateSignalCUDA(tx_samples, rx_samples_compensated, cor_i_start, cor_i_end, f_shifts)
#    log_A = np.log(A)
#    os.makedirs(f"results/DOPPLER_DELAY_TRIAGE", exist_ok=True)
#    pl.imsave(f"results/DOPPLER_DELAY_TRIAGE/{rx_filename}_log_A.png", log_A.T, vmax=log_A.max() * 0.8, vmin=log_A.max() * 0.4) 
#
#    os.makedirs(f"results/A", exist_ok=True)
#    np.save(open(f"results/A/{rx_filename}_A.npy", 'wb'), A)
    
    A = np.load(open(f"results/A/{rx_filename}_A.npy", 'rb'))
    sample_rate = rx_info['core:sample_rate'] / au.s
    rx_start_astrotime = at.Time(rx_info['dt:datetime'])
    print(rx_start_astrotime)
    log_A = np.log(A)

    doppler_values = f_shifts * au.Hz + doppler_start
    delay_values = cor_lags[cor_i_start:cor_i_end]

    NSIDE = 400
    NPIX = hp.nside2npix(NSIDE)
    
    v = np.array(hp.pix2vec(NSIDE, np.arange(NPIX))).T
    print(f"{v.shape=} BEFORE")
    # Crudely filter out the far hemisphere.
    v_near = v[:, 0] > 0
    v = v[v_near, :]
    print(f"{v.shape=} AFTER")
    
    dlt, lt = radar_surface_dlt(rx_start_astrotime, v)
    
    val_surface = np.zeros(NPIX)
    doppler_index = np.argmin((doppler_values[:, np.newaxis] - dlt * frequency)**2, axis=0)
    delay_index = np.argmin((delay_values[:, np.newaxis] - ((lt + 0.00005) * au.s + TX_START))**2, axis=0)
    val_surface[v_near] = log_A[doppler_index, delay_index]
    
    fig = pl.figure(figsize=(16,16))
    hp.orthview(val_surface, flip='geo', title='Moon Surface Value',
                fig=fig,
                #rot=srp_deg,
                half_sky=True,
               )
    hp.graticule()
    break
    
#    # Find the sub-rx point
#    srp, _ = sp.spkpos("STOCKERT", sp.str2et(rx_start_astrotime.utc.value), "MOON_ME", "CN+S", "MOON")
#    srp_rad = np.array(sp.reclat(srp)[1:])
#    print(f"{np.rad2deg(srp_rad)=}")
#    # Compute the doppler angle using the delta srp method
#    srp1, _ = sp.spkpos("STOCKERT", sp.str2et((rx_start_astrotime.utc + 0.1 * au.s).value), "MOON_ME", "CN+S", "MOON")
#    srp1_rad = np.array(sp.reclat(srp1)[1:])
#    dsrp_rad = srp1_rad - srp_rad
#    #doppler_angle = -np.arctan2(-dsrp_rad[1], -dsrp_rad[0] * np.cos(srp_rad[1])) * au.radian
#
#    y = math.sin(dsrp_rad[0]) * math.cos(srp1_rad[1])
#    x = (math.cos(srp_rad[1]) * math.sin(srp1_rad[1]) -
#         math.sin(srp_rad[1]) * math.cos(srp1_rad[1]) * math.cos(dsrp_rad[0]))
#    doppler_angle = math.atan2(y, x) * au.radian
#    print(f"  {doppler_angle=}")
#
#    ## HACK: use zeros for the SRP and doppler angle, for debugging....
#    #G = np.zeros((2000, 2000), dtype=np.float32) # TODO: use a smaller data rep?
#    #Gc = np.zeros(G.shape, dtype=np.int16)
#    #dopplerDelayToSphericalProjection(log_A.T, G, Gc,
#    #                                  1.0 / sample_rate.value,
#    #                                  0, 0,
#    #                                  0 * au.deg,
#    #)
#    #np.divide(G, Gc, out=G, where=Gc>0)  # Divide in place to save memory
#    #os.makedirs(f"results/ZERO_GLOBAL_TRIAGE", exist_ok=True)
#    #pl.imsave(f"results/ZERO_GLOBAL_TRIAGE/{rx_filename}_global.png", np.flipud(G))
#
#    if 'chan0' in rx_filename:   # HACK!
#        G = np.zeros((2000, 2000), dtype=np.float32) # TODO: use a smaller data rep?
#        Gc = np.zeros(G.shape, dtype=np.int16)
#    dopplerDelayToSphericalProjection(log_A.T,   # WHY TRANSPOSE? FIX?
#                                      G, Gc,
#                                      1.0 / sample_rate.value,
#                                      srp_rad[0], srp_rad[1],
#                                      -doppler_angle,
#    )
#    if 'chan1' in rx_filename:  # HACK
#        np.divide(G, Gc, out=G, where=Gc>0)  # Divide in place to save memory
#        os.makedirs(f"results/GLOBAL_TRIAGE", exist_ok=True)
#        pl.imsave(f"results/GLOBAL_TRIAGE/{rx_filename}_global.png", G)
#    
##    # Accumulate the image.
##    Go = G.copy()
##    np.divide(Go, Gc, out=Go, where=Gc>0)  # NOTE: this doesn't initialize Go! 
##    os.makedirs(f"results/GLOBAL_CUMULATIVE_TRIAGE", exist_ok=True)
##    pl.imsave(f"results/GLOBAL_CUMULATIVE_TRIAGE/{rx_filename}_global_cumulative.png", np.flipud(Go))

In [ ]:
if 0: # Test different methods for calculating the min/max doppler expected to be observed.
    # 1. Sample all over the surface of the moon, find the min and max doppler.
    # 2. Sample around the terminator, find the min and max doppler.
    # 3. Just sample the two 'limb points' (perpendicular to the sub radar point, in the direction of apparent rotation) (which should be on the terminator)
    
    rx_start_csptime = csp.str2et(rx_start_astrotime.utc.value)
    
    # Method 1: 
    #NSIDE = 200
    #NPIX = hp.nside2npix(NSIDE)
    #healpix_resol_deg = hp.nside2resol(NSIDE, arcmin=True) / 60
    #print(f"HEALPix {NSIDE=} {NPIX=} {healpix_resol_deg=:.2f}")
    #v = np.array(hp.pix2vec(NSIDE, np.arange(NPIX))).T
    #dlt, lt = radar_surface_dlt(rx_start_astrotime, v)
    print(f"HEALP {dlt.min():+0.016f} {dlt.max():+0.016f}")
    
    # Method 2:
    # We use UMBRAL for the terminator of last possible reflection.
    # 2a: Use the whole Earth as the 'light source'.
    target_epoch, observer_pos, v_term_e = csp.edterm("UMBRAL", "EARTH",     "MOON", rx_start_csptime, "MOON_ME", "CN", "STOCKERT", 1000)  # Treats the whole "EARTH" as the light source
    #print(target_epoch, observer_pos)
    # 2b: Just use the Dwingeloo antenna as the 'light source'.
    target_epoch, observer_pos, v_term_d = csp.edterm("UMBRAL", "DWINGELOO", "MOON", rx_start_csptime, "MOON_ME", "CN", "STOCKERT", 1000)  # Want to just use Dwingloo as the light source!
    #print(target_epoch, observer_pos)
    # To do so, we have to add the telescope radius (20m) to observatories.tf:
    #  BODY399999_RADII                    = (  0.02   0.02   0.02  )
    # And of course... I'm not sure that the magic of SPICE is actually figuring all this out!
    #print(v_term)
    dlt_term_e, lt_term_e = radar_surface_dlt(rx_start_astrotime, v_term_e)
    dlt_term_d, lt_term_d = radar_surface_dlt(rx_start_astrotime, v_term_d)
    print(f"EARTH {dlt_term_e.min():+0.016f} {dlt_term_e.max():+0.016f} DELTA_HEALP {dlt.min() - dlt_term_e.min():+0.016f} {dlt.max() - dlt_term_e.max():+0.016f}")
    print(f"DWING {dlt_term_d.min():+0.016f} {dlt_term_d.max():+0.016f} DELTA_HEALP {dlt.min() - dlt_term_d.min():+0.016f} {dlt.max() - dlt_term_d.max():+0.016f}")
    
    # I would expect the DWINGELOO range of doppler values to be *within* the EARTH range of doppler values. 
    # But the opposite seems to be the cause! Explanations:
    # a. The telescope radius trick didn't work, and SPICE is confused... 
    #    - but when I change the Dwingeloo radius to Earth radius, the results get closer!
    #      - (but not as close as I would expect?)
    # b. There are sampling artefacts
    #    - but the deltas stay roughly the same from 100 to 10000 points
    # c. I don't understand something about umbral/penumbral, or about the max doppler points (not on the terminator?), or about relativity?
    #    - ow my brain
    #    - ok, the Earth is big, so the earth-light terminator could be "past" the min-max speeds (as actually observed from Stockert)
    # ... guess I'll go with that, and consider Method 2b to be a faster and more precise replacement for Method 1.

In [ ]:
if 1:  # Debug: visualize the predicted doppler/delay on the observed doppler delay image.
    B = log_A.copy()
    #B = np.zeros(log_A.shape) + 1
    #doppler_values = f_shifts * au.Hz + doppler_start - 0.10 * au.Hz  # for the 03_11 HACK HACK HACK
    doppler_values = f_shifts * au.Hz + doppler_start  # No adjustment needed for the 06_21 data!
    delay_values = cor_lags[cor_i_start:cor_i_end]
    DOPPLER_PAD = 1
    DELAY_PAD = 2
    VALUE = log_A.max()
    #VALUE = 0
    
    for i in range(len(dlt_list)):
        dlt = dlt_list[i]
        lt = lt_list[i]
        doppler_index = np.argmin((doppler_values - dlt * frequency)**2)
        delay_index = np.argmin((delay_values - ((lt + 0.00005) * au.s + TX_START))**2)  # HACK
        B[(doppler_index-DOPPLER_PAD):(doppler_index+DOPPLER_PAD+1), (delay_index-DELAY_PAD):(delay_index+DELAY_PAD+1)] = VALUE
    
    ## Vectorized version -- slower (!?) and *much* more memory intensive!
    #doppler_indices = np.argmin((doppler_values - (dlt_surface * frequency)[:, np.newaxis])**2, axis=-1)
    #delay_indices = np.argmin((delay_values - (lt_surface * au.s + TX_START)[:, np.newaxis])**2, axis=-1)
    #B[doppler_indices, delay_indices] = VALUE
    
    pl.figure(figsize=(16,16))
    pl.imshow(B.T, #B.T * log_A.T, 
              extent=[
                  f_shifts[0],
                  f_shifts[-1],
                  cor_lags[cor_i_start].value,
                  cor_lags[cor_i_end].value,
              ],
              aspect='auto',
              interpolation='none',
              vmin=log_A.max() * 0.4,
              #vmax=log_A.max() * 0.8, 
             )
    
    #os.makedirs(f"{DATA_ROOT}/PREDICT_TRIAGE", exist_ok=True)
    #pl.savefig(f"{DATA_ROOT}/PREDICT_TRIAGE/{rx_chan0_sigmf_filename.split('/')[-1]}_doppler_grid.png")

# Appendix

In [ ]:
#if 0: # Estimate of the visible radius (in degrees) of the moon, given the range from the earth.
#    d = 385_000_000
#    r = 3_500_000
#    math.degrees(np.arcsin(np.sqrt(d**2 - r**2)/d))

In [ ]:
# Note: we calculate the doppler at the start and end of rx, not tx, because we will use it to
# correct rx_samples.
# There are a lot of different ways to compute this: astropy, jplHorizons, poliastro, spicey.

## SIMPLE
#def moonDopplerAndDelay_astropy(rx_time, tx_location, rx_location):
#    approx_moon_gcrs = ac.get_body("moon", rx_time, location=tx_location)
#    approx_light_travel_time = (approx_moon_gcrs.distance - MOON_RADIUS) / ak.c
#    
#    # Transmit "up" from Dwingeloo to the moon.
#    # NOTE: get_body returns the apparent position of the object at the observer's time and location,
#    # so for "up", we really want the earth-observed-from-moon direction, but we can only get moon-observed-from-earth.
#    # Since astropy get_body doesn't give radial velocity directly, we simply take the derivative of the range!
#    dt = 0.1 * au.s
#    up_moon_gcrs_1 = ac.get_body("moon", rx_time - approx_light_travel_time - dt, location=tx_location)
#    up_moon_gcrs_2 = ac.get_body("moon", rx_time - approx_light_travel_time + dt, location=tx_location)
#    up_radial_velocity = ((up_moon_gcrs_2.distance - up_moon_gcrs_1.distance) / (2 * dt)).to(au.m / au.s)
#    
#    # Receive "down" from the moon to Stockert. Note: we actually get the "up" path, just with a slight delay.
#    down_moon_gcrs_1 = ac.get_body("moon", rx_time - dt, location=rx_location)
#    down_moon_gcrs_2 = ac.get_body("moon", rx_time + dt, location=rx_location)
#    down_radial_velocity = ((down_moon_gcrs_2.distance - down_moon_gcrs_1.distance) / (2 * dt)).to(au.m / au.s)
#    
#    delay = (up_moon_gcrs_1.distance + down_moon_gcrs_2.distance - 2 * MOON_RADIUS) / ak.c
#    doppler = frequency * (up_radial_velocity + down_radial_velocity) / ak.c
#    return doppler, delay.to(au.s)

In [ ]:
#if 0: # Investigate the potential up-doppler/down-doppler asymmetry.
#    v_base = 300 * au.m / au.s
#    dv_limb = 3 * au.m / au.s
#    doppler_base = frequency * (v_base) / ak.c
#    doppler_inbound = frequency * (v_base - dv_limb) / ak.c
#    doppler_outbound = frequency * (v_base + dv_limb) / ak.c
#    print(doppler_base)
#    print(doppler_inbound)
#    print(doppler_outbound)
#    print(doppler_base - doppler_inbound)
#    print(doppler_outbound - doppler_base)
#    print(doppler_base - doppler_inbound - doppler_outbound + doppler_base)
#    # ... Conclusion: it ain't much! Negligible.

In [ ]:
## Reference CPU implementation. About 3 s/it. (seconds per iteration!)
#def correlateSignal(tx_samples, rx_samples):
#    # HACK: Hard-code search parameters for now...
#    echo_window = [5410000, 12300000]
#    delay_start = 2_805_000
#    delay_end = 2_840_000
#    f_shifts = range(-250, 251, 1)
#    #f_shifts = range(-650, -51, 1)
#    A = np.zeros((len(f_shifts), delay_end - delay_start))
#
#    for i in tqdm(range(len(f_shifts))):
#        tx_samples_shifted = scipy.signal.resample(tx_samples, 5_000_000 + f_shifts[i])
#        c_shifted = scipy.signal.correlate(rx_samples[echo_window[0]:echo_window[1]], tx_samples_shifted, mode='same')
#        A[i] = np.abs(c_shifted[delay_start:delay_end])
#
#    return A

In [ ]:
## Parallelized CPU implementation. About 1.5 it/s.
#import joblib
#
#def correlateSignalParallel(tx_samples, rx_samples):
#    # HACK: Hard-code parameters for now...
#    echo_window = [5410000, 12300000]
#    delay_start = 2_805_000
#    delay_end = 2_840_000
#    f_shifts = range(-250, 251, 1)
#    #f_shifts = range(-650, -51, 1)
#    
#    # Perform correlation for a single shift
#    def processShift(f_shift):
#        #tx_samples_shifted = scipy.signal.resample(tx_samples, len(tx_samples) + f_shift)
#        tx_samples_shifted = scipy.signal.resample(tx_samples, 5_000_000 + f_shift)
#        c_shifted = scipy.signal.correlate(rx_samples[echo_window[0]:echo_window[1]], tx_samples_shifted, mode='same')
#        correlation_slice = c_shifted[delay_start:delay_end]
#        return np.abs(correlation_slice)
#
#    # I find that using ~12 jobs on a 12 core Ryzen (which has 24 logical cpus) yields the best throughput.
#    results = joblib.Parallel(n_jobs=-4)(joblib.delayed(processShift)(f_shift) for f_shift in tqdm(f_shifts, desc="Correlating Shifts"))
#    A = np.vstack(results)
#    return A

In [ ]:
#def coarseTuneRoll(img, filename=None): # Coarse-tune the doppler centering by rolling to maximize left-right symmetry
#    best_col_offset = 0
#    best_col_offset_sum = 0
#    #img_h = img.T  # Whole image
#    # Extract a slice that corresponds to the limb being in position for img_mirror below.
#    #img_h = img[:, 3000:3200].copy().T  # For 1 MHz data
#    img_h = img[:, 1400:1600].copy().T  # For 250 kHz data
#    #pl.imshow(img_h)
#    offset_range = range(-200, 201)  # Depends on how confident we are in the doppler prediction!
#    for offset in offset_range:
#        img_tmp = np.roll(img_h, offset)
#        img_mirror = img_tmp[:, 400:600] * np.fliplr(img_tmp[:, -600:-400])
#        #img_mirror = img_tmp * np.fliplr(img_tmp)  # Whole row!
#        #pl.figure(figsize=(2, 3))
#        #pl.imshow(img_mirror)
#        #pl.title('' + str(offset))
#        total_sum = np.sum(img_mirror)
#        #print(offset, total_sum)
#        if total_sum > best_col_offset_sum:
#            best_col_offset_sum = total_sum
#            best_col_offset = offset
#    return best_col_offset

In [ ]:
#if 0: # Pixel-level search for central doppler frequency that leads to symmetric lobes.
##    best_col_offset = coarseTuneRoll(log_A)
##    print(f"{best_col_offset=}")
##    print(f"(one column frequency offset is {abs(f_shifts[0] - f_shifts[1])})")
##    best_freq_offset = f_shifts[int(f_shifts.shape[0] / 2) - best_col_offset] * au.Hz
##    print(f"{best_freq_offset=}")
##    best_col_doppler = doppler_start + best_freq_offset
##    print(f"{best_col_doppler=}")
##    
##    log_A_rolled = np.roll(log_A.T, best_col_offset).T
#    
#    if 0:  # Debug
#        pl.figure(figsize=(16,16))
#        pl.imshow(log_A_rolled.T + log_A_rolled.T[:, ::-1], aspect='auto')  
#        pl.gca().axvline(log_A.shape[0] / 2.0, linestyle="--",color="grey")
#
#    if 1:  # Result
#        #pl.figure(figsize=(16,16))
#        #pl.imshow(log_A_rolled.T, aspect='auto', vmax=log_A_rolled.max() * 0.8, vmin=log_A.max() * 0.4)
#        #pl.gca().axvline(log_A.shape[0] / 2.0, linestyle="--",color="grey")
#        pl.imsave(rx_chan0_sigmf_filename + "_log_A_rolled.png", log_A_rolled.T, vmax=log_A_rolled.max() * 0.8, vmin=log_A_rolled.max() * 0.4) 